In [3]:
import warnings
warnings.filterwarnings('ignore')

import sys
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# ROOT IMPORTS
# ============================================================

sys.path.append(
    os.path.abspath("..")
)

from constants import (
    NSDCC_CLEAN,
    NSDCC_COUNTY_COL,
    NSDCC_PERIOD_COL,
    IIT_WEIGHT,
    VLS_WEIGHT,
    HTS_WEIGHT,
    CGI_SCALE_MAX
)

In [4]:
df = pd.read_csv(NSDCC_CLEAN)

print(df.shape)

df.head()

(47, 51)


,period,county,MOH 731_HIV_TB_OnART_15-19_(F)_HV03-24,MOH 731_HIV_TB_OnART_15-19_(M)_HV03-23,MOH 731_HIV_TB_OnART_20-24_(F)_HV03-26,MOH 731_HIV_TB_OnART_20-24_(M)_HV03-25,MOH 731_HIV_TB_OnART_25+_(F)_HV03-28,MOH 731_HIV_TB_OnART_25+_(M)_HV03-27,adults_on_art,art_total_males,...,Actively on Treatment by the Beginning of 2025 female adult,iit_female,Actively on Treatment by the Beginning of 2025 Male,iit_male,adults_on_treatment,iit_count,iit_rate_pct,iit_rate_children_pct,iit_rate_male_pct,iit_rate_female_pct
0,2025,Baringo,82,70,164,92,3629,1701,5738,1863,...,524.214286,80.357143,280.00,42.714286,804.214286,123.071429,0.1530,0.131,0.153,0.153
1,2025,Bomet,234,160,416,175,7745,3616,12346,3951,...,524.214286,80.357143,280.00,42.714286,804.214286,123.071429,0.1530,0.131,0.153,0.153
2,2025,Bungoma,591,450,1235,400,20203,7717,30596,8567,...,932.500000,85.250000,425.75,36.250000,1358.250000,121.500000,0.0895,0.070,0.085,0.091
3,2025,Busia,543,467,940,425,23538,11203,37116,12095,...,932.500000,85.250000,425.75,36.250000,1358.250000,121.500000,0.0895,0.070,0.085,0.091
4,2025,Elgeyo Marakwet,83,68,149,55,3196,1368,4919,1491,...,524.214286,80.357143,280.00,42.714286,804.214286,123.071429,0.1530,0.131,0.153,0.153


In [5]:
df.columns.tolist()

['period',
 'county',
 'MOH 731_HIV_TB_OnART_15-19_(F)_HV03-24',
 'MOH 731_HIV_TB_OnART_15-19_(M)_HV03-23',
 'MOH 731_HIV_TB_OnART_20-24_(F)_HV03-26',
 'MOH 731_HIV_TB_OnART_20-24_(M)_HV03-25',
 'MOH 731_HIV_TB_OnART_25+_(F)_HV03-28',
 'MOH 731_HIV_TB_OnART_25+_(M)_HV03-27',
 'adults_on_art',
 'art_total_males',
 'art_total_females',
 'MOH 731_HTS_Tests _(F) (Including PMTCT)_ HV01-02',
 'MOH 731_HTS_Tests _(M)_ HV01-01',
 'hts_tested',
 'hts_tested_males',
 'hts_tested_females',
 'MOH 731_HTS_Positive_2-9 _(M)_ HV01-06',
 'MOH 731_HTS_Positive_2-9 _(F) (Including PMTCT)_ HV01-07',
 'MOH 731_HTS_Positive_10-14 _(F) (Including PMTCT)_HV01-09',
 'MOH 731_HTS_Positive_10-14 _(M)_HV01-08',
 'MOH 731_HTS_Positive_15-19 _(F) (Including PMTCT)_HV01-11',
 'MOH 731_HTS_Positive_15-19 _(M)_HV01-10',
 'MOH 731_HTS_Positive_20-24 _(M)_HV01-12',
 'MOH 731_HTS_Positive_20-24 _(F) (Including PMTCT)_HV01-13',
 'MOH 731_HTS_Positive_25+ _(M)_HV01-14',
 'MOH 731_HTS_Positive_25+ _(F) (Including PMTCT)_H

In [9]:
# creating HTS Positivity Rate 
if "hts_positivity_rate" not in df.columns:

    df["hts_positivity_rate"] = (
        df["hts_positive"] /
        df["hts_tested"]
    ).fillna(0)

In [7]:
df["iit_rate_yoy_change"] = 0

df["vls_rate_adult_yoy_change"] = 0

##### Engineering care Gap Index

In [16]:
# Convert percentages if needed
rate_cols = [
    "iit_rate_pct",
    "vls_rate_adult",
    "hts_positivity_rate"
]

for col in rate_cols:

    if df[col].max() > 1:
        df[col] = df[col] / 100

raw_cgi = (
    (IIT_WEIGHT * df["iit_rate_pct"]) +
    (VLS_WEIGHT * (1 - df["vls_rate_adult"])) +
    (HTS_WEIGHT * df["hts_positivity_rate"])
)

df["care_gap_index"] = (
    raw_cgi * CGI_SCALE_MAX
)

In [17]:
df["care_gap_index"].describe()

count    47.000000
mean      6.946324
std       1.639166
min       4.497835
25%       5.528375
50%       6.442406
75%       8.112314
max      11.366276
Name: care_gap_index, dtype: float64

In [18]:
# Engineering Art Coverage
if (
    "adults_on_art" in df.columns and
    "plhiv_estimate" in df.columns
):

    df["art_coverage"] = (
        df["adults_on_art"] /
        df["plhiv_estimate"]
    ).clip(0, 1).fillna(0)

else:

    print(
        "WARNING: Placeholder ART coverage used"
    )

    df["art_coverage"] = 0.5

In [19]:
df["art_coverage"].describe()

count    47.0
mean      0.5
std       0.0
min       0.5
25%       0.5
50%       0.5
75%       0.5
max       0.5
Name: art_coverage, dtype: float64

In [20]:
engineered_cols = [
    "iit_rate_yoy_change",
    "vls_rate_adult_yoy_change",
    "hts_positivity_rate",
    "care_gap_index",
    "art_coverage"
]

df[engineered_cols].head()

,iit_rate_yoy_change,vls_rate_adult_yoy_change,hts_positivity_rate,care_gap_index,art_coverage
0,0,0,0.006349,9.046978,0.5
1,0,0,0.011902,9.138043,0.5
2,0,0,0.011876,5.345510,0.5
3,0,0,0.012605,5.208106,0.5
4,0,0,0.007338,8.290762,0.5


In [24]:
county_profiles = (
    df.sort_values("period")
    .groupby("county")
    .last()
    .reset_index()
)

In [26]:
county_profiles.to_csv(
    "../data/processed/county_profiles.csv",
    index=False
)

print("county_profiles.csv saved successfully")

county_profiles.csv saved successfully


In [28]:
from src.feature_engineering import (
    run_feature_engineering
)

In [29]:
df = pd.read_csv(
    "../data/processed/nsdcc_clean.csv"
)